In [1]:
#format the book
from __future__ import division, print_function
%matplotlib inline
import sys
sys.path.insert(0, '..')
import book_format
book_format.set_style()

# 将多元方程化为一元情形

多元卡尔曼滤波（multivariate Kalman filter）方程与一元滤波器的方程并不相似。然而，若状态与测量都是一维的，这些方程就退化为一元方程。本节将帮助你建立对卡尔曼滤波方程实际在做什么的直观理解。读本节并非理解全书其余部分的必要条件，但我建议仔细阅读，因为它应使后续内容更容易理解。

下面是预测（prediction）的多元方程。

$$
\begin{aligned}
\mathbf{\bar{x}} &= \mathbf{F x} + \mathbf{B u} \\
\mathbf{\bar{P}} &= \mathbf{FPF}^\mathsf{T} + \mathbf Q
\end{aligned}
$$

对一元问题，状态 $\mathbf x$ 只有一个变量，因此是 $1\times 1$ 矩阵。运动 $\mathbf{u}$ 也是 $1\times 1$ 矩阵。因此 $\mathbf{F}$ 与 $\mathbf B$ 也必须是 $1\times 1$ 矩阵，即它们都是标量，可写为

$$\bar{x} = Fx + Bu$$

这里变量不加粗，表示它们不是矩阵或向量。

状态转移很简单——下一状态与当前状态相同，故 $F=1$。运动转移同样，$B=1$。于是

$$x = x + u$$

等价于上一章的高斯方程

$$ \mu = \mu_1+\mu_2$$

希望一般过程已经清楚，下面我会稍快一些。我们有

$$\mathbf{\bar{P}} = \mathbf{FPF}^\mathsf{T} + \mathbf Q$$

同样，由于状态只有一个变量，$\mathbf P$ 与 $\mathbf Q$ 也是 $1\times 1$ 矩阵，可当作标量，得到

$$\bar{P} = FPF^\mathsf{T} + Q$$

已知 $F=1$。标量的转置仍是该标量，故 $F^\mathsf{T} = 1$。于是

$$\bar{P} = P + Q$$

等价于高斯方程

$$\sigma^2 = \sigma_1^2 + \sigma_2^2$$

这证明：在维数为 1 时，多元预测方程与一元方程做的是同样的数学运算。

下面是更新（update）步的方程：

$$
\begin{aligned}
\mathbf{K}&= \mathbf{\bar{P}H}^\mathsf{T} (\mathbf{H\bar{P}H}^\mathsf{T} + \mathbf R)^{-1} \\
\textbf{y} &= \mathbf z - \mathbf{H \bar{x}}\\
\mathbf x&=\mathbf{\bar{x}} +\mathbf{K\textbf{y}} \\
\mathbf P&= (\mathbf{I}-\mathbf{KH})\mathbf{\bar{P}}
\end{aligned}
$$

同上，所有矩阵都变成标量。$H$ 定义如何从位置转换为测量。两者都是位置，无需转换，故 $H=1$。代入已知值并一步化为标量形式。$1\times 1$ 矩阵的逆就是该值的倒数，因此把矩阵求逆写成除法。

$$
\begin{aligned}
K &=\frac{\bar{P}}{\bar{P} + R} \\
y &= z - \bar{x}\\
x &=\bar{x}+Ky \\
P &= (1-K)\bar{P}
\end{aligned}
$$

在继续证明之前，请看看这些方程，认识它们实现的简单概念。残差（residual）$y$ 无非是测量（measurement）减去预测（prediction）。增益（gain）$K$ 根据我们对上次预测与测量的确信程度进行缩放。新状态 $x$ 在旧 $x$ 基础上加上缩放后的残差。最后，根据对测量的确信程度更新不确定性。从算法上看，这应与上一章所做的完全一致。

下面完成代数证明。回忆一元更新步方程为：

$$
\begin{aligned}
\mu &=\frac{\sigma_1^2 \mu_2 + \sigma_2^2 \mu_1} {\sigma_1^2 + \sigma_2^2}, \\
\sigma^2 &= \frac{1}{\frac{1}{\sigma_1^2} + \frac{1}{\sigma_2^2}}
\end{aligned}
$$

这里设 $\mu_1$ 为状态 $x$，$\mu_2$ 为测量 $z$。于是 $\sigma_1^2$ 为状态不确定性 $P$，$\sigma_2^2$ 为测量噪声（measurement noise）$R$。代入得

$$\begin{aligned} \mu &= \frac{Pz + Rx}{P+R} \\
\sigma^2 &= \frac{1}{\frac{1}{P} + \frac{1}{R}}
\end{aligned}$$

先处理 $\mu$。多元情形对应方程为

$$
\begin{aligned}
x &= x + Ky \\
&= x + \frac{P}{P+R}(z-x) \\
&= \frac{P+R}{P+R}x + \frac{Pz - Px}{P+R} \\
&= \frac{Px + Rx + Pz - Px}{P+R} \\
&= \frac{Pz + Rx}{P+R}
\end{aligned}
$$

再看 $\sigma^2$。多元情形对应方程为

$$ 
\begin{aligned}
P &= (1-K)P \\
&= (1-\frac{P}{P+R})P \\
&= (\frac{P+R}{P+R}-\frac{P}{P+R})P \\
&= (\frac{P+R-P}{P+R})P \\
&= \frac{RP}{P+R}\\
&= \frac{1}{\frac{P+R}{RP}}\\
&= \frac{1}{\frac{R}{RP} + \frac{P}{RP}} \\
&= \frac{1}{\frac{1}{P} + \frac{1}{R}}
\quad\blacksquare
\end{aligned}
$$

我们已证明：只有一个状态变量时，多元方程与一元方程等价。本节末尾再提一点——我略过了 $H=1$ 与 $F=1$ 的断言。一般情况下它们并不成立。例如数字温度计可能以伏特给出测量，需转换为温度，用 $H$ 完成该转换。为保持说明简洁，我省略了这一点。把该推广加入上方方程、重做代数，仍会得到相同结果。\\\